# WLS 02. Refresh Shared Regression View

Updates the Delta Sharing regression view without dropping it.
The added `MODEL` row exposes the overall WLS model p-value (`prob_f`) through `p_value`.

In [ ]:
# COMMAND ----------

SHARED_REGRESSION_VIEW = (
    'sandbox.t_online_voc_analysis.tv_voc_weighted_regression_intellytics'
)
COEF_TABLE = 'sandbox.z_jungryo_lee.tv_voc_weighted_regression_intellytics'
MODEL_TABLE = 'sandbox.z_jungryo_lee.tv_voc_weighted_regression_model_intellytics'

spark.sql(f"""
ALTER VIEW {SHARED_REGRESSION_VIEW} AS
SELECT
  segment_col,
  segment_value,
  group_dim,
  group_key,
  y_feature,
  x_feature,
  coef,
  p_value,
  t_value,
  y_obs,
  x_obs,
  abs_coef,
  driver_rank,
  is_driver,
  weighted_corr,
  r_squared,
  adj_r_squared,
  data_created_dt
FROM {COEF_TABLE}

UNION ALL

SELECT
  segment_col,
  segment_value,
  group_dim,
  group_key,
  y_feature,
  'MODEL' AS x_feature,
  CAST(NULL AS DOUBLE) AS coef,
  prob_f AS p_value,
  CAST(NULL AS DOUBLE) AS t_value,
  y_obs,
  CAST(NULL AS BIGINT) AS x_obs,
  CAST(NULL AS DOUBLE) AS abs_coef,
  CAST(NULL AS BIGINT) AS driver_rank,
  0 AS is_driver,
  CAST(NULL AS DOUBLE) AS weighted_corr,
  r_squared,
  adj_r_squared,
  data_created_dt
FROM {MODEL_TABLE}
""")

print(f'updated shared view: {SHARED_REGRESSION_VIEW}')

In [ ]:
# COMMAND ----------

display(
    spark.table(SHARED_REGRESSION_VIEW)
    .where("x_feature = 'MODEL'")
    .select(
        'segment_value',
        'group_dim',
        'group_key',
        'y_feature',
        'p_value',
        'r_squared',
        'adj_r_squared',
        'y_obs',
    )
    .orderBy('segment_value', 'group_dim', 'group_key', 'y_feature')
)